In [136]:
import requests
from bs4 import BeautifulSoup as bs
import warnings
import re
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore", category = Warning)

In [112]:
malagasy = []
anglais = []
francais = []

In [72]:
URL_dep = "https://tenymalagasy.org/bins/homePage"
URL =  "https://tenymalagasy.org"
# response = requests.get(URL)
response = requests.get(URL_dep, verify=False)
print('Status code: ', response.status_code)

Status code:  200


In [73]:
soup = bs(response.content, "html.parser")

In [74]:
links = soup.find_all("a")

In [75]:
# Initialiser les listes
text_numbers = []
full_urls = []

# Parcourir les liens
for link in links:
    text = link.get_text(strip=True)
    href = link.get('href')
    
    # Vérifier si le texte est un nombre avec regex
    if re.match(r'^\d[\d\s]*\d$', text.replace(" ", "")):
        text_numbers.append(text)
        full_urls.append(URL + href)

# Afficher les résultats
# print("Nombres extraits :", text_numbers)
# print("URLs complètes :", full_urls)

Nombres extraits : ['109\xa0000', '34\xa0000', '15\xa0000', '8\xa0000', '8\xa0000', '1\xa0000', '1\xa0000', '500']
URLs complètes : ['https://tenymalagasy.org/bins/alphaLists?lang=mg', 'https://tenymalagasy.org/bins/alphaLists?lang=fr', 'https://tenymalagasy.org/bins/alphaLists?lang=en', 'https://tenymalagasy.org/bins/proverbIndex', 'https://tenymalagasy.org/bins/acronyms', 'https://tenymalagasy.org/bins/geoLists', 'https://tenymalagasy.org/bins/imageLists#arrows', 'https://tenymalagasy.org/bins/patroLists']


In [76]:
response_niv2 = requests.get(full_urls[0], verify=False)
print('Status code: ', response_niv2.status_code)

Status code:  200


In [77]:
soup = bs(response_niv2.content, "html.parser")

print("Page title: " , soup.title.string)

Page title:  Rakibolana sy Rakipahalalana malagasy : alphaLists


In [84]:
center = soup.find("center")

table_link = center.find_all("table", class_ = "menuLink")

table_link = table_link[0]
link_2 = table_link.find_all("a")

# link_2 = link_2[0]

# Initialiser les listes
text2 = []
full_urls2 = []

# Parcourir les liens
for link in link_2:
    # text = link.get_text(strip=True)
    href = link.get('href')
    href_total = URL + href
    text_numbers.append(text)
    full_urls2.append(href_total)

# Afficher les résultats
# print("URLs complètes :", full_urls2)

In [137]:
for i, j in tqdm(enumerate(full_urls2), total = len(full_urls2), desc="Traitement des URLs"):
    response_niv3 = requests.get(full_urls2[i], verify=False)
    soup = bs(response_niv3.content, "html.parser")

    # print("Page title: " , soup.title.string)


    center = soup.find('center')

    tds_80 = center.select('td[width="80%"]')

    tds = tds_80[0]

    table = tds.find_all("table", class_ = "menuLink")

    if table is None: 
        print("Table non trouvée")

    table = table[0]

    rows = table.find_all("tr")

    for row in rows:
        cells = row.find_all("td")

        if len(cells) >= 3:
            malagasy_text = cells[0].get_text(strip=True)
            centre_text = cells[1].get_text(strip=True)
            francais_text = cells[2].get_text(strip=True)

        malagasy.append(malagasy_text)
        anglais.append(centre_text)
        francais.append(francais_text)


Traitement des URLs: 100%|███████████████████████████████████████████████████████████| 141/141 [07:03<00:00,  3.00s/it]


In [134]:
data = {
    "Malagasy" : malagasy,
    "Francais": francais,
    "Anglais": anglais
}

df = pd.DataFrame(data)

df.drop_duplicates(inplace=True)
df.to_json("dictionnaire malagasy.json", orient = 'records', indent = 4)
df.to_csv("dictionnaire malagasy(;).csv", index=False, sep=";")
df.to_csv("dictionnaire malagasy(,).csv", index=False, sep=",")